**第9章　為替レートと外国為替市場：決定木学習による固定相場制度分析**

　本例では、決定木学習を用いて、固定相場制度の離脱予測を行います。標本データはブラジル・レアル、ベトナム・ドンの対USドルレートを用います。まず、統計分析と数値計算のためのpandas、グラフ作成のためのmatplotlib、そしてsklearnから決定木学習のためのDecisionTreeClassifierをはじめとする3つのライブラリをインポートします。データセットは、1995年7月から2026年5月までの、ブラジルレアルの対USドルレート（real_bi）、ブラジルの物価上昇率：前年同月比―米国の物価上昇率前年同月比（Pus_bz）、ブラジルのデット・サービス・レシオ（bz_dsr）、ブラジルの経常収支対GDP比（bz_is_balance）のデータから構成されます。

In [3]:
#[1]ライブラリの読み込み
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from sklearn.tree import DecisionTreeClassifier #決定木学習のためのライブラリ
from sklearn.model_selection import train_test_split #訓練・テストデータ分割ライブラリ
from sklearn.metrics import accuracy_score #予測検証のライブラリ

　続いて、本演算に用いるCSVデータを読み込みます。データはオンラインストレージGithubに9_fixed_regime_data.csvというファイル名で保存されています。読み込み後は時間（月次）情報が入力されている列をインデックス化しておきます。この1995年7月から2026年5月までの371カ月間、ブラジルレアルの対USドルレート（real_bi）が、10%以上の下落（切り下げ）を経験するとこのデータは「=1」、10%未満の変動であれば「=0」と定義しています。また、ブラジル―米国のインフレ率格差（Pus_bz）が10％を超えた場合も「=1」、10%未満の変動であれば「=0」、ブラジルのデット・サービス・レシオ（bz_dsr:　公的対外債務年返済額÷輸出額）が20%超が「=1」、20%未満が「=0」と定義しています。経常収支の対GDP比（bz_is_balance）は▲3%以下の場合「=1」、▲3%よりも大きければ「=0」と定義します。

In [ ]:
#[2]データの取得
url = "https://github.com/nagamamo/data/blob/main/9_fixed_regime_data.csv?raw=true"#Git-hubからCSVデータの入手
df = pd.read_csv(url)#データフレームの作成
df["t"] = pd.to_datetime(df["t"]) #日時をインデックスの変換のためdatetimeへ変換
df = df.set_index("t")#日時をインデックスへ変換
df.head()

　本例では、ブラジルのインフレ率データは一カ月前、二カ月前、三カ月前のラグ付き変数を使用します。ラグ付きデータの変換は下記により新変数としてデータフレームに追加されます。

In [5]:
#[3]インフレ率データのラグ変数への変換
df['Pus_bz_1'] = df['Pus_bz'].shift(1)
df['Pus_bz_2'] = df['Pus_bz'].shift(2)
df['Pus_bz_3'] = df['Pus_bz'].shift(3)
df['Pus_vn_1'] = df['Pus_vn'].shift(1)
df['Pus_vn_2'] = df['Pus_vn'].shift(2)
df['Pus_vn_3'] = df['Pus_vn'].shift(3)

　次に、ブラジルのインフレ率（米国インフレ率との差）、デット・サービス・レシオ、経常収支対GDP比の二値データを説明変数、ブラジルレアルの対USドルレートの二値データを被説明変数として定義します。第４節の３行目は、Xとｙのデータを７０％：３０％の比率で訓練データとテストデータに分割する指示を意味しています。

In [6]:
#[4]インフレ率データのラグ変数への変換
X = df[['Pus_bz_1', 'Pus_bz_2','Pus_bz_3','bz_dsr','bz_is_balance']]
y = df['real_bi']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

　最後にレアル切り下げが、どの程度の確率により発生しうるのかを算出するモデルを推計します。１行目では、決定木分類モデルがmodelと称する実証方程式であることを定義します。そして２行目においてデータを用いてフィッティングを行い、３行目では２行目の推計結果とテストデータを用いて予測値を算出します。

In [7]:
#[5]決定木モデルの構築と学習
model = DecisionTreeClassifier(random_state=42) #決定木学習モデルの定義
model.fit(X_train, y_train) #フィッティング
y_pred = model.predict(X_test) #テストデータによる予測値の算出
print("適合率:", accuracy_score(y_test, y_pred)) #実績値と予測値の適合比率の算出

適合率: 0.9732142857142857
